# Cleaning Netflix Dataset
**Author:** Wajid Ali Saleem Chaudhry  
**Project:** https://roadmap.sh/projects/cleaning-netflix-dataset  
**Dataset:** Netflix Movies and TV Shows (`data/netflix_titles.csv`)

## C1 — Setup & Initial Exploration

In [1]:
# Author:      Wajid Ali Saleem Chaudhry
# Description: Imports for the Netflix dataset cleaning notebook.

import pandas as pd

In [2]:
# --- Load Dataset ---
df = pd.read_csv("data/netflix_titles.csv")


In [3]:
# --- Shape & Schema ---
print(df.shape)
df.info()

(8807, 12)
<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [4]:
# --- Descriptive Statistics ---
display(df.describe())
df.describe(include="object")

,release_year
count,8807.000000
mean,2014.180198
std,8.819312
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


C:\Users\Chaud\AppData\Local\Temp\claude\ipykernel_21840\2339099979.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,show_id,type,title,director,cast,country,date_added,rating,duration,listed_in,description
count,8807,8807,8807,6173,7982,7976,8797,8803,8804,8807,8807
unique,8807,2,8807,4528,7692,748,1767,17,220,514,8775
top,s1,Movie,Dick Johnson Is Dead,Rajiv Chilaka,David Attenborough,United States,"January 1, 2020",TV-MA,1 Season,"Dramas, International Movies","Paranormal activity at a lush, abandoned prope..."
freq,1,6131,1,19,19,2818,109,3207,1793,362,4


In [5]:
# --- Preview ---
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [6]:
# --- Missing Value Survey ---
missing = df.isnull().sum()
missing_percent = (missing / len(df))*100

summary = pd.DataFrame({
    'count': missing,
    'percent': missing_percent.round(2)
})

summary[summary['count'] > 0].sort_values('count',ascending=False) 

,count,percent
director,2634,29.91
country,831,9.44
cast,825,9.37
date_added,10,0.11
rating,4,0.05
duration,3,0.03


## C2 — Handle Missing Values

In [7]:
# --- Inspect Before Deciding ---
summary


,count,percent
show_id,0,0.00
type,0,0.00
title,0,0.00
director,2634,29.91
cast,825,9.37
country,831,9.44
date_added,10,0.11
release_year,0,0.00
rating,4,0.05
duration,3,0.03


In [8]:
# --- director (high % missing) ---
# Many titles have no listed director (e.g. TV shows, collaborations).
# Dropping these rows would lose too much data.

df["director"] = df["director"].fillna("Unknown")

In [9]:
# --- cast (high % missing) ---
# Same reasoning as director.
df["cast"] = df["cast"].fillna("Unknown")

In [10]:
# --- country (moderate % missing) ---
# Fill with mode (most common origin country) — a reasonable default
# given the distribution is heavily US-skewed.

country_mode = df["country"].mode()[0]
print("mode:", country_mode)
df["country"] = df["country"].fillna(country_mode)

mode: United States


In [11]:
# --- rating (few missing) ---
# Rating has a clear dominant value; fill with mode.

rating_mode = df["rating"].mode()[0]
print("mode:", rating_mode)
df["rating"] = df["rating"].fillna(rating_mode)

mode: TV-MA


In [12]:
# --- date_added / duration (very few missing) ---
# So few rows are affected that dropping them is safe —
# we lose almost nothing and avoid inventing dates or runtimes.

df = df.dropna(subset=["date_added", "duration"])

In [13]:
# --- Verify ---
df.isnull().sum()

show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64

## C3 — Fix Data Types

In [14]:
# --- Split duration into numeric value + unit ---
# "90 min"    → duration_value=90, duration_unit="min"
# "2 Seasons" → duration_value=2,  duration_unit="Seasons"

extracted = df["duration"].str.extract(r"^(\d+)\s+(.+)$")
df["duration_value"] = extracted[0].astype(int)
df["duration_unit"]  = extracted[1]

df[["duration", "duration_value", "duration_unit"]].head()

,duration,duration_value,duration_unit
0,90 min,90,min
1,2 Seasons,2,Seasons
2,1 Season,1,Season
3,1 Season,1,Season
4,2 Seasons,2,Seasons


In [15]:
# --- Convert date_added to datetime ---
# Strip leading/trailing whitespace first; the raw values often have
# a leading space (e.g. " January 1, 2020").

df["date_added"] = pd.to_datetime(
    df["date_added"].str.strip(), format="%B %d, %Y"
)

df["date_added"].dtype

dtype('<M8[us]')

In [16]:
# --- Verify final dtypes ---
df.dtypes

show_id                      str
type                         str
title                        str
director                     str
cast                         str
country                      str
date_added        datetime64[us]
release_year               int64
rating                       str
duration                     str
listed_in                    str
description                  str
duration_value             int64
duration_unit                str
dtype: object

## C4 — Export Cleaned Dataset

In [17]:
# --- Export ---
df.to_csv("data/netflix_titles_cleaned.csv", index=False)
print(f"Exported {len(df)} rows, {len(df.columns)} columns.")

Exported 8794 rows, 14 columns.
